In [ ]:
#| default_exp harness

# harness

> getting the record out of whichever agent is running

An adapter turns one harness payload into a list of `Scribe` calls. It writes nothing, so it can be read without a disk.

In [ ]:
#| export
import json, os, re, sys
from importlib.resources import files
from pathlib import Path

from fastcore.basics import AttrDict
from fastcore.foundation import L

from panjika.core import Home, now
from panjika.core import Scribe, action_for, repo_facts

## What an adapter returns

A plan: a session id, where it ran, and the acts to apply.

In [ ]:
#| export
def plan(session='', start='', **acts):
    "Empty plan for one session."
    return AttrDict(session=str(session or ''), start=str(start or ''), acts=L(), **acts)


def act(p, do, **kw):
    "Add one call to a plan. `do` names a `Scribe` method."
    p.acts.append(AttrDict(do=do, **kw))
    return p


def first_of(d, *keys, default=''):
    "The first key present and non-empty in `d`."
    for k in keys:
        v = (d or {}).get(k)
        if v not in (None, '', [], {}): return v
    return default

## Claude Code

One JSON object on stdin per event.

`PATH_KEYS` is where a path hides in a tool's arguments; `TARGET_KEYS` is what else is worth naming when there is none.

In [ ]:
#| export
PATH_KEYS = ('file_path', 'filePath', 'notebook_path', 'notebookPath', 'path', 'filename', 'file')

TARGET_KEYS = ('command', 'url', 'pattern', 'query', 'prompt', 'description', 'code')


def tool_target(args):
    "The one thing a tool call was pointed at: a path, else a command, a url, a pattern."
    return str(first_of(args, *PATH_KEYS, *TARGET_KEYS))


def tool_path(args):
    "The file a tool call names, or `''`."
    return str(first_of(args, *PATH_KEYS))

In [ ]:
#| export
def claude_code(payload):
    "One Claude Code hook payload as a plan. See the hooks reference for the field names."
    p = payload or {}
    event = str(p.get('hook_event_name') or '')
    out = plan(p.get('session_id'), p.get('cwd'))
    args = p.get('tool_input') or {}
    tool = str(p.get('tool_name') or '')
    if event == 'SessionStart':
        act(out, 'begin', harness='claude-code', model=str(p.get('model') or ''),
            agent=str(p.get('agent_type') or ''), parent=str(p.get('agent_id') and p.get('session_id') or ''))
    elif event == 'UserPromptSubmit':
        text = str(first_of(p, 'user_input', 'prompt'))
        act(out, 'note', text=text)
        act(out, 'write', kind='session', prompt=text[:2000])
    elif event in ('PostToolUse', 'PostToolUseFailure'):
        ok = event == 'PostToolUse'
        act(out, 'step', tool=tool, target=tool_target(args), ok=ok, args=args,
            output=p.get('tool_response'), summary='' if ok else 'the tool failed')
        path = tool_path(args)
        if ok and path and action_for(tool) == 'write':
            act(out, 'touch', path=path, action='edit')
    elif event in ('Stop', 'SubagentStop'):
        last = str(p.get('last_assistant_message') or '')
        if last: act(out, 'note', text=last[:4000])
    elif event == 'SessionEnd':
        act(out, 'end', status='done', reason=str(p.get('reason') or ''))
    return out

## Codex

Claude Code's event names, delivered the same way. Verified against the [hook reference](https://developers.openai.com/codex/hooks).

Three things differ. There is no failure event, so an error is read out of `tool_response`. `apply_patch` names its files inside the patch. A subagent reports its parent's `session_id` beside its own `agent_id`.

The legacy `notify` route fires once per turn and never per tool call, is kebab-case, and arrives as an argument with stdin closed, so it goes through `panjika record`.

In [ ]:
#| export
CODEX_EDITS = ('apply_patch', 'Edit', 'Write')
_PATCH_FILE = re.compile(r'^\*\*\* (?:Add|Update|Delete) File: (.+)$', re.M)


def patch_paths(text):
    "The files an `apply_patch` envelope names."
    return [m.group(1).strip() for m in _PATCH_FILE.finditer(str(text or ''))]


def codex_paths(tool, args):
    "The files one Codex tool call moved."
    if tool in CODEX_EDITS:
        body = first_of(args, 'command', 'input', 'patch', 'content')
        if (found := patch_paths(body)): return found
    path = tool_path(args)
    return [path] if path and action_for(tool) == 'write' else []


def _codex_failed(response):
    "Whether a `tool_response` reports an error. Codex has no separate failure event."
    if isinstance(response, dict):
        return bool(response.get('isError') or response.get('is_error') or response.get('error'))
    return False


def codex(payload):
    "One Codex lifecycle-hook payload as a plan."
    p = payload or {}
    event = str(p.get('hook_event_name') or '')
    out = plan(p.get('session_id'), p.get('cwd'))
    args = p.get('tool_input') or {}
    tool = str(p.get('tool_name') or '')
    agent = str(p.get('agent_type') or p.get('agent_id') or '')
    if event in ('SessionStart', 'SubagentStart'):
        act(out, 'begin', harness='codex', model=str(p.get('model') or ''), agent=agent,
            parent=str(p.get('session_id') or '') if p.get('agent_id') else '')
    elif event == 'UserPromptSubmit':
        text = str(first_of(p, 'prompt', 'user_input', 'user_prompt'))
        act(out, 'note', text=text)
        act(out, 'write', kind='session', prompt=text[:2000], origin='human')
    elif event == 'PostToolUse':
        resp = p.get('tool_response')
        ok = not _codex_failed(resp)
        act(out, 'step', tool=tool, target=tool_target(args), ok=ok, args=args, output=resp,
            summary='' if ok else 'the tool reported an error')
        if ok:
            for path in codex_paths(tool, args): act(out, 'touch', path=path, action='edit')
    elif event in ('Stop', 'SubagentStop'):
        text = str(first_of(p, 'last_assistant_message', 'last-assistant-message'))
        if text: act(out, 'note', text=text[:4000])
    elif event == 'SessionEnd':
        act(out, 'end', status='done', reason=str(p.get('reason') or ''))
    return out


def codex_notify(payload):
    "One legacy Codex `notify` payload as a plan."
    p = payload or {}
    if str(p.get('type') or '') != 'agent-turn-complete': return plan('', '')
    out = plan(first_of(p, 'thread-id', 'turn-id'), str(p.get('cwd') or ''))
    act(out, 'begin', harness='codex', model='', title=str(p.get('client') or ''))
    for text in (p.get('input-messages') or []):
        act(out, 'write', kind='session', prompt=str(text)[:2000], origin='human')
    if (last := p.get('last-assistant-message')): act(out, 'note', text=str(last)[:4000])
    act(out, 'end', status='turn-complete', turn=str(p.get('turn-id') or ''))
    return out

## Ramabana

The turn record `Agent._remember` already builds, handed over whole. An `Act` on its own is one step.

`at` is when the turn ran, and the session is stamped with it. Ramabana replays its own history, and a session stamped at ingest would sort as the newest thing that ever happened.

In [ ]:
#| export
def ramabana(payload):
    "One Ramabana turn record as a plan, or one `Act` on its own as one step."
    p = payload or {}
    out = plan(first_of(p, 'session', 'session_id'), first_of(p, 'cwd', 'root'))
    if not any(p.get(k) for k in ('activity', 'prompt', 'reply', 'tool', 'model', 'error')):
        return out
    if p.get('tool') and 'activity' not in p:
        act(out, 'step', tool=p['tool'], target=_act_target(p), ok=bool(p.get('ok', True)),
            secs=float(p.get('secs') or 0), summary=str(p.get('summary') or ''),
            args=p.get('args'), output=p.get('detail'))
        if _act_path(p): act(out, 'touch', path=_act_path(p), action='edit')
        return out
    usage, at = p.get('usage') or {}, p.get('at')
    act(out, 'write', kind='session', harness='ramabana', model=str(p.get('model') or ''),
        prompt=str(p.get('prompt') or '')[:2000], status=str(p.get('state') or 'done'),
        at=at, started=at,
        tokens_in=usage.get('input'), tokens_out=usage.get('output'), cost=usage.get('cost'))
    for a in (p.get('activity') or ()):
        act(out, 'step', tool=str(a.get('tool') or ''), target=_act_target(a),
            ok=bool(a.get('ok', True)), secs=float(a.get('secs') or 0),
            summary=str(a.get('summary') or ''), args=a.get('args'), output=a.get('detail'))
        if a.get('ok', True) and _act_path(a): act(out, 'touch', path=_act_path(a), action='edit')
    if p.get('reply'): act(out, 'note', text=str(p['reply'])[:4000])
    if p.get('error'): act(out, 'note', text=f"failed: {p['error']}")
    return out


def _act_target(a): return tool_target(a.get('args') or {}) or str(a.get('summary') or '')


def _act_path(a):
    "The file a Ramabana act names, when the tool it ran was one that writes."
    path = tool_path(a.get('args') or {})
    return path if path and action_for(a.get('tool') or '') == 'write' else ''


## Anything else

A payload already in the ledger's shape: one act, or a list under `acts`.

`ADAPTERS` is every harness this package can read. A new one is a function and a line here.

In [ ]:
#| export
def generic(payload):
    "A payload that already speaks the ledger's own shape, as one act or a list under `acts`."
    p = dict(payload or {})
    out = plan(first_of(p, 'session', 'session_id'), first_of(p, 'cwd', 'start'))
    acts = p.get('acts')
    if acts is None:
        do = p.pop('do', None)
        for k in ('session', 'session_id', 'cwd', 'start', 'harness_name'): p.pop(k, None)
        if not do and not p: return out
        acts = [{'do': do or 'note', **p}]
    for a in acts:
        a = dict(a)
        act(out, a.pop('do', 'note'), **a)
    return out


ADAPTERS = {'claude-code': claude_code, 'codex': codex, 'codex-notify': codex_notify, 'ramabana': ramabana,
            'generic': generic}

## Making the calls

A hook that fails must not take the harness down, so failures are swallowed to stderr. A payload no adapter reads writes nothing and makes no ledger.

`begin` records the repository, root and branch. Ramabana uses `write` instead, so `ingest` stamps the same facts on a session record that lacks them. Those are the facts now, not during the turn.

In [ ]:
#| export
def ingest(payload, adapter='generic', home=None, start='.', harness=''):
    "Apply one payload to a ledger. Returns the plan that was applied."
    build = ADAPTERS.get(str(adapter), generic)
    p = build(payload)
    if not p.acts:
        p.wrote, p.home = 0, ''
        return p
    sc = Scribe(home=home, session=p.session, start=p.start or start)
    if not sc.home.exists: sc.home.init()
    facts = None
    for a in p.acts:
        do = a.pop('do')
        if do == 'begin': a.setdefault('harness', harness or str(adapter))
        elif do == 'write' and a.get('kind') == 'session' and 'branch' not in a:
            if facts is None: facts = repo_facts(sc.start)
            a = {**facts, **a}
        getattr(sc, do)(**a)
    p.wrote, p.home, p.session = len(p.acts), str(sc.home.path), sc.session
    return p


def hook(adapter='generic', stream=None, home=None, start=None):
    "Read one payload from `stream` and record it. Never raises; failures go to stderr."
    try:
        raw = (stream or sys.stdin).read()
        payload = json.loads(raw) if raw.strip() else {}
    except Exception as e:
        print(f'panjika: unreadable payload ({e})', file=sys.stderr)
        return None
    if not payload: return None
    try: return ingest(payload, adapter, home, start or payload.get('cwd') or os.getcwd())
    except Exception as e:
        print(f'panjika: {type(e).__name__}: {e}', file=sys.stderr)
        return None

## Trying it

In [ ]:
import subprocess, tempfile
from fastcore.test import test_eq
from panjika.core import Ledger

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('X = 1\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')
Home(d/'.panjika').init()

def cc(**kw): return ingest({'session_id': 'cc-1', 'cwd': str(d), **kw}, 'claude-code', d/'.panjika')

cc(hook_event_name='SessionStart', model='opus-5')
cc(hook_event_name='UserPromptSubmit', user_input='bump the constant')
(d/'app.py').write_text('X = 2\n')
cc(hook_event_name='PostToolUse', tool_name='Edit',
   tool_input={'file_path': str(d/'app.py')}, tool_response='ok')
cc(hook_event_name='PostToolUseFailure', tool_name='Bash',
   tool_input={'command': 'pytest'}, tool_response='2 failed')
cc(hook_event_name='SessionEnd', reason='clear')

row = Ledger(d/'.panjika').session('cc-1')
test_eq((row.harness, row.model, row.status), ('claude-code', 'opus-5', 'done'))
test_eq(row.prompt, 'bump the constant')
test_eq((row.steps_ok, row.steps_fail), (1, 1))
test_eq([t.path for t in row.files], ['app.py'])

In [ ]:
ingest({'session': 'rb-1', 'cwd': str(d), 'model': 'sonnet', 'prompt': 'tidy the imports',
        'reply': 'done, two files', 'usage': {'input': 1840, 'output': 96},
        'activity': [{'tool': 'search_code', 'args': {'query': 'import'}, 'ok': True, 'secs': 0.2},
                     {'tool': 'edit_file', 'args': {'path': str(d/'app.py')}, 'ok': True, 'secs': 0.4},
                     {'tool': 'run_shell', 'args': {'command': 'pytest -q'}, 'ok': False}]},
       'ramabana', d/'.panjika')
row = Ledger(d/'.panjika').session('rb-1')
test_eq((row.harness, row.model), ('ramabana', 'sonnet'))
test_eq(row.branch, 'main')
test_eq((row.n_steps, row.steps_fail), (3, 1))
test_eq([t.path for t in row.files], ['app.py'])

In [ ]:
ingest({'session': 'sh-1', 'cwd': str(d), 'do': 'begin', 'harness': 'a shell script',
        'prompt': 'regenerate the fixtures'}, 'generic', d/'.panjika')
ingest({'session': 'sh-1', 'cwd': str(d), 'do': 'step', 'tool': 'make', 'target': 'fixtures'},
       'generic', d/'.panjika')
test_eq([s.harness for s in Ledger(d/'.panjika').sessions()],
        ['a shell script', 'ramabana', 'claude-code'])

Ramabana never calls `begin`, so nothing else knows when its turn ran.

In [ ]:
JAN = 1768460400.0
def _turn(at, prompt, **kw):
    return {'session': 'rb-when', 'cwd': str(d), 'at': at, 'model': 'sonnet', 'prompt': prompt,
            'usage': {'input': 10, 'output': 2}, **kw}

ingest(_turn(JAN, 'round the vat'), 'ramabana', d/'.panjika')
ingest(_turn(JAN + 300, 'and add a test'), 'ramabana', d/'.panjika')

row = Ledger(d/'.panjika').session('rb-when')
test_eq(row.started, JAN)
test_eq(row.prompt, 'round the vat')
test_eq(row.seconds, 300.0)

assert Ledger(d/'.panjika').sessions()[0].session != 'rb-when' 

### What a payload looks like

Claude Code delivers one JSON object on stdin per event. Codex uses the same events, differing in the three ways below.

In [ ]:
p = claude_code({'hook_event_name': 'PostToolUse', 'session_id': 'cc-1', 'cwd': '/repo',
                 'tool_name': 'Edit', 'tool_input': {'file_path': 'charges.py'},
                 'tool_response': 'ok'})
test_eq([a.do for a in p.acts], ['step', 'touch'])
test_eq(p.acts[1].path, 'charges.py')

envelope = '*** Begin Patch\n*** Update File: charges.py\n@@\n-a\n+b\n*** End Patch\n'
p = codex({'hook_event_name': 'PostToolUse', 'session_id': 'cdx-1', 'cwd': '/repo',
           'model': 'gpt-5.6', 'tool_name': 'apply_patch', 'tool_use_id': 'c1',
           'tool_input': {'command': envelope}, 'tool_response': 'done'})
test_eq([a.path for a in p.acts if a.do == 'touch'], ['charges.py'])

p = codex({'hook_event_name': 'PostToolUse', 'session_id': 'cdx-1', 'tool_name': 'Bash',
           'tool_input': {'command': 'pytest'}, 'tool_response': {'isError': True}})
assert not p.acts[0].ok
test_eq([a for a in p.acts if a.do == 'touch'], [])

p = codex_notify({'type': 'agent-turn-complete', 'thread-id': 'th-1', 'turn-id': '99',
                  'cwd': '/repo', 'input-messages': ['rename foo to bar'],
                  'last-assistant-message': 'renamed'})
test_eq(p.session, 'th-1')
test_eq([a for a in p.acts if a.do in ('step', 'touch')], [])

A hook given a broken payload leaves the harness alone.

In [ ]:
import io
test_eq(hook('claude-code', io.StringIO('{not json')), None)
test_eq(hook('claude-code', io.StringIO('')), None)
test_eq(hook('claude-code', io.StringIO('{}')), None)

test_eq(ingest({'session_id': 'x', 'hook_event_name': 'PreCompact'}, 'claude-code').wrote, 0)

## Installing the hooks

`settings.json` for Claude Code, `hooks.json` for Codex, `post-commit` for git. Codex runs no hook until you review it with `/hooks`. Both documents are merged: a panjika entry is replaced, anything else is left alone.

`CODEX_EVENTS` is Codex's lifecycle events and what each matches. `CODEX_SNIPPET` is the legacy `notify` route for a Codex too old to have hooks.

In [ ]:
#| export
CC_EVENTS = {'SessionStart': '', 'UserPromptSubmit': '', 'SessionEnd': '',
             'PostToolUse': '*', 'PostToolUseFailure': '*'}

POST_COMMIT = """#!/bin/sh
# Written by `panjika install --git`. Links each commit to the agent sessions that earned it.
panjika link-commit "$(git rev-parse HEAD)" >/dev/null 2>&1 || true
"""

CODEX_EVENTS = {'SessionStart': '', 'UserPromptSubmit': '', 'SessionEnd': '',
                'PostToolUse': '.*', 'SubagentStart': '', 'SubagentStop': ''}

CODEX_TRUST = ("Codex will not run a hook until you have reviewed it. Run `/hooks` in Codex, "
               "read the entry, and trust it.")

CODEX_SNIPPET = """# ~/.codex/config.toml
# Only for a Codex without lifecycle hooks. One record per turn, no tool calls.
notify = ["panjika", "record", "--adapter", "codex-notify"]
"""


def codex_hooks(path='.codex/hooks.json', command='panjika hook codex'):
    "The Codex hooks document with panjika's hooks merged into it."
    p = Path(path)
    try: doc = json.loads(p.read_text()) if p.exists() else {}
    except Exception: doc = {}
    if not isinstance(doc, dict): doc = {}
    hooks = doc.setdefault('hooks', {})
    for event, matcher in CODEX_EVENTS.items():
        groups = [g for g in hooks.get(event, [])
                  if not any('panjika' in str(h.get('command', ''))
                             for h in (g.get('hooks') or []))]
        group = {'hooks': [{'type': 'command', 'command': command, 'timeout': 10}]}
        if matcher: group['matcher'] = matcher
        hooks[event] = groups + [group]
    return doc


def cc_settings(path='.claude/settings.json', command='panjika hook claude-code'):
    "The Claude Code settings document with panjika's hooks merged into it."
    p = Path(path)
    try: doc = json.loads(p.read_text()) if p.exists() else {}
    except Exception: doc = {}
    if not isinstance(doc, dict): doc = {}
    hooks = doc.setdefault('hooks', {})
    for event, matcher in CC_EVENTS.items():
        entries = [e for e in hooks.get(event, [])
                   if not any('panjika' in str(h.get('command', ''))
                              for h in (e.get('hooks') or []))]
        entry = {'hooks': [{'type': 'command', 'command': command, 'timeout': 10}]}
        if matcher: entry['matcher'] = matcher
        hooks[event] = entries + [entry]
    return doc

## The skill

`SKILL.md` ships inside the package. `install` writes it into each agent's skill folder, and `[project.entry-points.pyskills]` publishes the same text. A pyskill body carries no frontmatter, so `skill_body` opens it with the description instead.

In [ ]:
#| export
SKILL_DIRS = ('.claude/skills/panjika', '.agents/skills/panjika', '.codex/skills/panjika')


def skill_md():
    "The packaged `SKILL.md`, frontmatter and all."
    return (files('panjika')/'SKILL.md').read_text(encoding='utf-8')


def frontmatter(text):
    "The `name` and `description` of a skill document, and the body after them."
    if not text.startswith('---\n'): return {}, text
    end = text.find('\n---', 3)
    if end < 0: return {}, text
    meta, key = {}, ''
    for line in text[4:end].splitlines():
        if line[:1] not in (' ', '\t') and ':' in line:
            key, _, v = line.partition(':')
            key = key.strip()
            meta[key] = v.strip().lstrip('>|').strip()
        elif key: meta[key] = f'{meta[key]} {line.strip()}'.strip()
    return meta, text[end + 4:].lstrip('\n')


def skill_body(text=None):
    "A skill document as a pyskill body: its description first, then the prose."
    meta, body = frontmatter(skill_md() if text is None else text)
    desc = ' '.join((meta.get('description') or '').split())
    return f'{desc}\n\n{body}' if desc else body


def write_skill(root='.', dirs=SKILL_DIRS):
    "Write the packaged skill into each agent's skill folder under `root`. Returns the paths."
    body, out = skill_md(), L()
    for d in dirs:
        p = Path(root)/d/'SKILL.md'
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(body)
        out.append(str(p))
    return out

In [ ]:
#| export
def install(root='.', claude_code=True, git=True, codex=True, skill=True, home=None):
    "Write the hook configuration and the skill into `root`. Returns what was written and what was not."
    root = Path(root)
    out, ledger = AttrDict(wrote=L(), skipped=L(), codex=CODEX_SNIPPET,
                           note=CODEX_TRUST), Home(home, root)
    ledger.init()
    out.home = str(ledger.path)
    if claude_code:
        p = root/'.claude'/'settings.json'
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(cc_settings(p), indent=2) + '\n')
        out.wrote.append(str(p))
    if codex:
        p = root/'.codex'/'hooks.json'
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(codex_hooks(p), indent=2) + '\n')
        out.wrote.append(str(p))
    if skill: out.wrote += write_skill(root)
    if git:
        from panjika.core import git_root
        gr = git_root(root)
        if gr is None: out.skipped.append('post-commit: not inside a git repository')
        else:
            p = Path(gr)/'.git'/'hooks'/'post-commit'
            if p.exists() and 'panjika' not in p.read_text():
                out.skipped.append(f'post-commit: {p} exists and is not ours')
            else:
                p.parent.mkdir(parents=True, exist_ok=True)
                p.write_text(POST_COMMIT)
                p.chmod(0o755)
                out.wrote.append(str(p))
    return out

In [ ]:
out = install(d, home=d/'.panjika')
out = install(d, home=d/'.panjika')
doc = json.loads((d/'.claude'/'settings.json').read_text())
for event in CC_EVENTS:
    entries = [h for e in doc['hooks'][event] for h in e['hooks'] if 'panjika' in h['command']]
    test_eq(len(entries), 1)
assert (Path(d)/'.git'/'hooks'/'post-commit').exists()

In [ ]:
doc = json.loads((d/'.claude'/'settings.json').read_text())
doc['hooks']['PostToolUse'].append({'matcher': 'Write', 'hooks': [{'type': 'command', 'command': 'ruff format'}]})
(d/'.claude'/'settings.json').write_text(json.dumps(doc))
install(d, home=d/'.panjika')
doc = json.loads((d/'.claude'/'settings.json').read_text())
commands = [h['command'] for e in doc['hooks']['PostToolUse'] for h in e['hooks']]
test_eq(sorted(commands), ['panjika hook claude-code', 'ruff format'])

In [ ]:
test_eq(sorted(Path(p).parent.parent.parent.name for p in write_skill(d)),
        ['.agents', '.claude', '.codex'])
for rel in SKILL_DIRS: test_eq((d/rel/'SKILL.md').read_text(), skill_md())

meta, body = frontmatter(skill_md())
test_eq(sorted(meta), ['description', 'name'])
test_eq(meta['name'], 'panjika')
assert not body.startswith('---')
test_eq(skill_body('---\nname: x\n---\n\nbody\n'), 'body\n')
test_eq(skill_body('no frontmatter\n'), 'no frontmatter\n')
assert skill_body().startswith(meta['description'][:40])

In [ ]:
doc = json.loads((d/'.codex'/'hooks.json').read_text())
doc['hooks']['PostToolUse'].append(
    {'matcher': 'apply_patch', 'hooks': [{'type': 'command', 'command': 'codex-fmt'}]})
(d/'.codex'/'hooks.json').write_text(json.dumps(doc))
res = install(d, home=d/'.panjika')

doc = json.loads((d/'.codex'/'hooks.json').read_text())
for event in CODEX_EVENTS:
    entries = [h for g in doc['hooks'][event] for h in g['hooks'] if 'panjika' in h['command']]
    test_eq(len(entries), 1)
test_eq(sorted(h['command'] for g in doc['hooks']['PostToolUse'] for h in g['hooks']),
        ['codex-fmt', 'panjika hook codex'])
assert 'reviewed it' in res.note

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()